# 01 — Coleta e qualidade dos dados

Pré-projeto *Preço do Café e da Soja × Dólar × Clima*. Este notebook executa (ou reaproveita) a coleta de cada fonte e inspeciona os dados brutos. Toda a lógica está em `src/`; aqui apenas chamamos as funções.

| Fonte | Técnica | Módulo |
|---|---|---|
| Notícias Agrícolas | Web scraping (requests + BeautifulSoup) | `src/coleta/noticias_agricolas.py` |
| CEPEA/ESALQ | Planilha (download manual) | `src/coleta/cepea.py` |
| Banco Central (SGS) | API REST | `src/coleta/bcb.py` |
| NASA POWER | API REST | `src/coleta/nasa_power.py` |
| INMET | Download de arquivos | `src/coleta/inmet.py` |
| Yahoo Finance | API (yfinance) | `src/coleta/mercado_internacional.py` |

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import config
from src.analise.estilo import aplicar_estilo_matplotlib

aplicar_estilo_matplotlib()
pd.set_option("display.max_columns", 50)

## Coleta
Defina `COLETAR = True` para baixar novamente. O backfill do histórico do Notícias Agrícolas é longo (~3 h) e deve ser rodado pelo terminal: `python -m src.coleta.noticias_agricolas --backfill`.

In [2]:
COLETAR = False
if COLETAR:
    from src.pipeline import coleta
    coleta(backfill=False, inmet=True)

## Dados brutos

In [3]:
cotacoes = pd.read_csv(config.RAW_NA / "cotacoes.csv", parse_dates=["data"])
print(cotacoes.groupby("produto")["data"].agg(["min", "max", "count"]))
cotacoes.tail()

               min        max  count
produto                             
cafe    2020-01-02 2026-11-19   1713
soja    2020-01-02 2026-11-19   1712


,data,produto,preco,variacao_pct,fonte,coletado_em
3420,2026-11-17,soja,167.43,0.82,noticias_agricolas,2026-11-21T09:50:34
3421,2026-11-18,cafe,1682.78,-0.31,noticias_agricolas,2026-11-21T09:50:31
3422,2026-11-18,soja,167.94,0.30,noticias_agricolas,2026-11-21T09:50:34
3423,2026-11-19,cafe,1709.06,1.56,noticias_agricolas,2026-11-21T09:50:31
3424,2026-11-19,soja,168.69,0.45,noticias_agricolas,2026-11-21T09:50:34


In [4]:
dolar = pd.read_csv(config.RAW_BCB / "dolar.csv", parse_dates=["data"])
ipca = pd.read_csv(config.RAW_BCB / "ipca.csv", parse_dates=["data"])
dolar.describe()

,data,dolar
count,1728,1728.000000
mean,2023-06-12 03:18:20,5.268796
min,2020-01-02 00:00:00,4.021300
25%,2021-09-21 18:00:00,5.061775
50%,2023-06-13 12:00:00,5.251400
75%,2025-02-27 06:00:00,5.485925
max,2026-11-19 00:00:00,6.208600
std,NaN,0.334782


In [5]:
nasa = pd.concat([pd.read_csv(p, parse_dates=["data"]) for p in sorted(config.RAW_NASA.glob("*.csv"))])
nasa.replace(-999, pd.NA).groupby("regiao").describe().T.head(15)

regiao          cerrado_mineiro       matas_de_minas         sul_de_minas
data count                 2517                 2517                 2517
     mean   2023-06-12 00:00:00  2023-06-12 00:00:00  2023-06-12 00:00:00
     min    2020-01-01 00:00:00  2020-01-01 00:00:00  2020-01-01 00:00:00
     25%    2021-09-21 00:00:00  2021-09-21 00:00:00  2021-09-21 00:00:00
     50%    2023-06-12 00:00:00  2023-06-12 00:00:00  2023-06-12 00:00:00
     75%    2025-03-02 00:00:00  2025-03-02 00:00:00  2025-03-02 00:00:00
     max    2026-11-21 00:00:00  2026-11-21 00:00:00  2026-11-21 00:00:00

In [6]:
estacoes = config.RAW_INMET / "estacoes_selecionadas.csv"
pd.read_csv(estacoes) if estacoes.exists() else "INMET ainda não coletado"

,regiao,ano,estacao,codigo,lat,lon,distancia_km,arquivo
0,sul_de_minas,2020,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2020.csv
1,cerrado_mineiro,2020,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2020.csv
2,matas_de_minas,2020,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2020.csv
3,sul_de_minas,2021,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2021.csv
4,cerrado_mineiro,2021,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2021.csv
5,matas_de_minas,2021,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2021.csv
6,sul_de_minas,2022,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2022.csv
7,cerrado_mineiro,2022,PATROCINIO,A523,-18.996684,-46.985935,5.9,cerrado_mineiro_A523_2022.csv
8,matas_de_minas,2022,MANHUACU,A556,-20.263333,-42.182778,15.6,matas_de_minas_A556_2022.csv
9,sul_de_minas,2023,VARGINHA,A515,-21.566389,-45.404167,3.2,sul_de_minas_A515_2023.csv


## Tratamento, integração e relatório de qualidade
Gera `data/processed/base_integrada.csv` e `reports/qualidade.md` (lacunas, outliers por IQR, validação NASA × INMET e Notícias Agrícolas × CEPEA).

In [7]:
from src.pipeline import tratamento
base = tratamento()
base.shape

09:51:57 INFO    src.tratamento.limpeza: interim/dolar.csv: 1728 linhas


09:51:57 INFO    src.tratamento.limpeza: interim/ipca.csv: 82 linhas


09:51:57 INFO    src.coleta.cepea: CEPEA cafe: 1713 observacoes


09:51:57 INFO    src.coleta.cepea: CEPEA soja: 1713 observacoes


09:51:57 INFO    src.tratamento.limpeza: interim/precos.csv: 3436 linhas


09:51:57 INFO    src.tratamento.limpeza: interim/internacional.csv: 1735 linhas


... (4 outputs left out)

09:51:58 INFO    src.coleta.cepea: CEPEA cafe: 1713 observacoes


09:51:58 INFO    src.coleta.cepea: CEPEA soja: 1713 observacoes


09:51:58 INFO    src.tratamento.qualidade: reports/qualidade.md gerado


(1728, 67)

In [8]:
from IPython.display import Markdown
Markdown((config.REPORTS / "qualidade.md").read_text(encoding="utf-8"))

# Relatório de qualidade dos dados

Gerado a partir de `data/processed/base_integrada.csv` (1728 dias úteis).

## Cobertura por variável

| variavel | inicio | fim | observacoes | faltantes_pct |
|---|---|---|---|---|
| preco_cafe | 2020-01-02 | 2026-11-19 | 1728 | 0.0 |
| preco_soja | 2020-01-02 | 2026-11-19 | 1728 | 0.0 |
| dolar | 2020-01-02 | 2026-11-19 | 1728 | 0.0 |
| cafe_ice_usd_saca | 2020-01-02 | 2026-11-19 | 1728 | 0.0 |
| soja_cbot_usd_saca | 2020-01-02 | 2026-11-19 | 1728 | 0.0 |
| ipca_mensal | 2020-01-02 | 2026-10-30 | 1715 | 0.8 |
| chuva_mm | 2020-01-02 | 2026-11-18 | 1727 | 0.1 |
| temp_med | 2020-01-02 | 2026-11-18 | 1727 | 0.1 |

## Maiores lacunas — preco_cafe

| de | ate | dias |
|---|---|---|
| 2020-02-21 | 2020-02-26 | 5 |
| 2021-02-12 | 2021-02-17 | 5 |
| 2022-02-25 | 2022-03-02 | 5 |
| 2023-02-17 | 2023-02-22 | 5 |
| 2024-02-09 | 2024-02-14 | 5 |

## Outliers de variação diária (IQR, k=3) — cafe

4 dias atípicos de 1727. Maiores:


| data | variacao_1d_pct |
|---|---|
| 2026-07-06 | 9.24 |
| 2021-07-22 | 7.06 |
| 2024-05-02 | -6.88 |
| 2021-07-21 | 6.41 |


Verificar se coincidem com eventos reais (ex.: geada de jul/2021) antes de remover.

## Maiores lacunas — preco_soja

| de | ate | dias |
|---|---|---|
| 2020-02-21 | 2020-02-26 | 5 |
| 2021-02-12 | 2021-02-17 | 5 |
| 2022-02-25 | 2022-03-02 | 5 |
| 2023-02-17 | 2023-02-22 | 5 |
| 2024-02-09 | 2024-02-14 | 5 |

## Outliers de variação diária (IQR, k=3) — soja

9 dias atípicos de 1727. Maiores:


| data | variacao_1d_pct |
|---|---|
| 2021-01-05 | 5.2 |
| 2020-11-12 | -5.09 |
| 2022-10-03 | -4.41 |
| 2020-12-09 | 4.31 |
| 2021-06-30 | 4.2 |
| 2022-04-01 | -4.14 |
| 2020-09-21 | 3.9 |
| 2020-11-09 | -3.82 |
| 2020-11-10 | 3.77 |


Verificar se coincidem com eventos reais (ex.: geada de jul/2021) antes de remover.

## Validação: Notícias Agrícolas × CEPEA

| produto | dias_em_comum | dif_media | dif_max |
|---|---|---|---|
| cafe | 1708 | 0.0 | 0.0 |
| soja | 1707 | 0.0004335090802577675 | 0.7400000000000091 |

## Validação: NASA POWER × INMET

| regiao | dias_em_comum | corr_chuva_mensal | corr_temp_med_diaria | corr_temp_min_diaria | vies_temp_min_C |
|---|---|---|---|---|---|
| cerrado_mineiro | 2465 | 0.947 | 0.845 | 0.843 | 1.85 |
| matas_de_minas | 2465 | 0.896 | 0.893 | 0.862 | 0.56 |
| sul_de_minas | 2465 | 0.855 | 0.931 | 0.928 | 0.79 |


In [9]:
base.isna().mean().sort_values(ascending=False).head(15).round(3)

chuva_180d                   0.062
soja_var_fut_63d             0.036
cafe_var_fut_63d             0.036
chuva_90d_cerrado_mineiro    0.032
chuva_90d_matas_de_minas     0.032
chuva_90d_sul_de_minas       0.032
chuva_90d                    0.032
anomalia_chuva_90d           0.032
soja_mm60                    0.025
cafe_mm60                    0.025
dolar_var_21d                0.012
soja_var_21d                 0.012
cafe_var_21d                 0.012
cafe_var_fut_21d             0.012
soja_var_fut_21d             0.012
dtype: float64